# modular-diffusion: stages, growth, content on averaged weights

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/modular-diffusion/blob/v0.5.0/colab/stages.ipynb)

How many stages of the noise level a model should be cut into, and where the cuts go (`geolip.modular_diffusion.experiments.stages`): copies of a shared model that continue its training run on one stage each, tails trained alone on a frozen shared stem, and a model grown one block at a time. Where the class has to be read, and class experts against two controls of equal compute (`experiments.content`). Heads solved in closed form on averaged weights, and class heads added to a finished model by a solve alone (`experiments.closed_forms`). First of all, a check of whether a busy CPU beside a job on the card changes that job's last digits.

1. install at a release tag and print the environment;
2. the data with its content hash, the finished runs this batch starts from (fetched from the Hub), a smoke run of the picture score and of the new paths, and the size of the job list;
3. the repeat check in three short phases, then the job list; each finished job is shipped to the Hub at once; a progress line a minute; the summary; then the machine is given back.

A stopped session resumes by running the cells again: finished jobs keep their markers under `OUT`, and on a new machine the jobs already on the Hub are fetched back first.

In [ ]:
# C1 - install at a tag (extras on the name); torch is not a dependency, so the runtime's torch stays
%pip install -q "modular-diffusion[data,hub,judge,guard] @ git+https://github.com/AbstractEyes/modular-diffusion@v0.5.0"
import os, sys, time, json, shutil, threading, subprocess
import torch, geolip.modular_diffusion as MD
from geolip.modular_diffusion import environment
assert tuple(int(x) for x in MD.__version__.split(".")) >= (0, 5, 0), "RESTART REQUIRED: Runtime > Restart session, then run C1 again"
environment.check(None)
print("card:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (choose a GPU runtime)", "| cpus:", os.cpu_count())
OUT = "/content/stages"              # the output root; keep it the same to resume
os.makedirs(OUT, exist_ok=True)

The data comes from the Hub as parquet files and becomes one npz of uint8 arrays with a content hash. The finished runs this batch starts from are fetched from the results repository: the reference model and the model that never reads the class, seeds 0 and 1, each with its averaged weights. The smoke run trains the reference for 60 steps and scores it (the score is checked against a second implementation and the class reader on the test pictures), then runs the new stage paths for a few steps.

In [ ]:
# C2 - the data, the runs this batch starts from, a smoke run, the job list's size
from huggingface_hub import snapshot_download
from geolip.modular_diffusion import data, score, linker
from geolip.modular_diffusion.beds import PicturesBed
from geolip.modular_diffusion.experiments import time_split, content, closed_forms, stages
NPZ = f"{OUT}/data/cifar10_32.npz"
REF = f"{OUT}/data/reference_inception.npz"     # the reference statistics: computed once here, read by every job
if not os.path.exists(NPZ):
    snapshot_download(repo_id="uoft-cs/cifar10", repo_type="dataset", allow_patterns=["plain_text/*.parquet"], local_dir=f"{OUT}/parquet")
    data.cifar_npz(parquet_dir=f"{OUT}/parquet/plain_text", out=NPZ)
EXPECTED_CONTENT_HASH = "be50ca799cbe6795ff13bca750f53f16c974b9f4903082fc06031b9719700016"
got = data.npz_content_hash(NPZ)
print("content hash", got, "| expected", EXPECTED_CONTENT_HASH or "(not filled)")
assert not EXPECTED_CONTENT_HASH or got == EXPECTED_CONTENT_HASH, "the npz is not the expected data"

REPO = "AbstractPhil/geolip-multirotary-diffusion-collective"   # the results repository ("" in C3 = nothing shipped)
SOURCES = ["plain_s0", "plain_s1", "blind_s0", "blind_s1"]       # finished runs of the content experiment, with averaged weights
hub_dir = snapshot_download(repo_id=REPO, allow_patterns=[f"modular_diffusion/content/{t}/*" for t in SOURCES]
                            + ["modular_diffusion/time_split/plain_s0/result.json"], local_dir=f"{OUT}/hub")
SRC = f"{hub_dir}/modular_diffusion/content"
for t in SOURCES:
    assert os.path.exists(f"{SRC}/{t}/result.json") and os.path.isdir(f"{SRC}/{t}/model_avg999"), f"the finished run {t} is not in the results repository"
FIRST = json.load(open(f"{hub_dir}/modular_diffusion/time_split/plain_s0/result.json"))["loss_over_levels"]
print("the number the reference must reproduce when it runs alone:", repr(FIRST))

SMOKE = f"{OUT}/smoke/plain"
smoke = content.main(variant="plain", seed=0, data=NPZ, out=SMOKE, device="cuda", scale=0.01, score=1, score_n=5000, ref=REF, reader=1)
assert "score" in smoke["score"], smoke["score"]
READER_CHECK = smoke["score"].get("reader_check")     # written by the job that computes the reference file: this one
print("class reader on the test pictures:", READER_CHECK, "| on the smoke model's samples:", (smoke["score"].get("class_read") or {}).get("accuracy"))
from torchmetrics.image.fid import FrechetInceptionDistance
bed = PicturesBed(NPZ, "cuda")
model = linker.link(f"{SMOKE}/model", device="cuda")
fake = score.sample_pictures(model, bed, 5000, steps=50, batch=1000, device="cuda")
real = bed.train_pictures()
fid = FrechetInceptionDistance(feature=2048, antialias=False).to("cuda")     # the same resize as the library's judge
for i in range(0, real.shape[0], 1000):
    fid.update(score.to_pictures(real[i:i + 1000], bed).to("cuda"), real=True)
for i in range(0, fake.shape[0], 1000):
    fid.update(fake[i:i + 1000].to("cuda"), real=False)
other = float(fid.compute())
mine = smoke["score"]["score"]
print(f"picture score: library {mine:.4f} | torchmetrics, the same resize {other:.4f} (relative difference {abs(mine - other) / other:.2e}) (5,000 pictures)")
del model, bed, real, fake, fid
torch.cuda.empty_cache()

# the new paths for a few steps each: a shared model with its trainer state, three copies that continue it, a tail and a
# grown block on its frozen stem, class heads solved on the smoke model
S = f"{OUT}/smoke"
common = dict(seed=0, data=NPZ, device="cuda", scale=0.01)
stages.main(variant="source", out=f"{S}/source", total=4000, score=0, **common)
sm_copies = stages.main(variant="copies", out=f"{S}/copies", source=f"{S}/source", total=2000, cuts="0.25,0.65", score=1, score_n=1000,
                        ref=REF, reader=1, score_last=0, score_avg="tail100", **common)
sm_tails = stages.main(variant="tails", out=f"{S}/tails", stem_from=SMOKE, stem_model="model", total=6000, cuts="0.25,0.65", score=0, **common)
sm_grow = stages.main(variant="grow", out=f"{S}/grow", stem_from=SMOKE, stem_model="model", blocks=2, steps_each=2000, score=0, **common)
sm_heads = closed_forms.main(variant="class_heads", out=f"{S}/class_heads", source=SMOKE, batches=5, score=0, seed=0, data=NPZ, device="cuda")
print("smoke of the new paths: copies", sm_copies["loss_over_levels"], "| tails", sm_tails["loss_over_levels"],
      "| grown", sm_grow["loss_over_levels"], "| class heads", sm_heads["loss_over_levels"])
INSTRUMENTS = {"reader_check": READER_CHECK, "smoke_class_read": smoke["score"].get("class_read"),
               "score_check": {"library": mine, "second_same_resize": other, "pictures": 5000},
               "first_run_loss": FIRST, "loads": {}}

# the job list: (tag, experiment, arguments, the jobs it waits for, its cost in reference-model steps); {R} = the runs folder
SCORE = 1700                                   # one scoring of 10,000 pictures, in reference-model steps
WIDTHS = "g3,g8,g20,tail100"                   # four averages kept beside the last weights: the check of the average's width
# the repeat check. Two phases with the default settings, each alone on the machine: the reference by itself (it should
# reproduce the first run to the last digit), then the same job beside a load on every CPU. A third phase beside the
# same load with det=on. Every later job runs with det=on: by default the backward pass of the attention kernel is not
# deterministic on a card; with the switch on it takes its deterministic algorithm, and a job repeats to the last digit
ALONE = [("repeat_alone_s0", "content", "variant=plain seed=0 avg=g8", [], 6000 + 2 * SCORE)]
LOADED = [("repeat_load_s0", "content", "variant=plain seed=0 score=0", [], 6000)]
FIXED_LOADED = [("ref_load_s0", "content", "variant=plain seed=0 avg=g8 score=0", [], 6000)]
# the reference with det=on at seeds 0 and 1, four averages kept beside the last weights (the check of the average's
# width), and a 300-step run in warn mode, which names what is not deterministic by default
J = [(f"ref_s{s}", "content", f"variant=plain seed={s} avg={WIDTHS}", [], 6000 + 5 * SCORE) for s in (0, 1)]
J += [("repeat_warn_s0", "content", "variant=plain seed=0 det=warn scale=0.05 score=0", [], 300)]
# the shared model at 4,000 steps with its trainer state, and the 4-block model with its average (the frozen stem)
J += [(f"source_s{s}", "stages", f"variant=source seed={s} total=4000 score=0", [], 4000) for s in (0, 1)]
J += [(f"plain_d4_avg_s{s}", "time_split", f"variant=plain_d4 seed={s} avg=g8 score_last=0", [], 3500 + SCORE) for s in (0, 1)]
CUTS = {"k1": "", "k2": "0.411", "k3": "0.25,0.65", "k3w": "0.333,0.667", "k3g": "0.461,0.772",
        "k5": "0.105,0.288,0.552,0.827", "k8": "0.064,0.145,0.262,0.411,0.588,0.764,0.908"}
COPY = "source_model=model total=2000 score_last=0 score_avg=tail100"
for s in (0, 1):
    J += [(f"copies_{k}_s{s}", "stages", f"variant=copies seed={s} source={{R}}/source_s{s} cuts={c} {COPY}", [f"source_s{s}"], 2000 + SCORE)
          for k, c in CUTS.items()]
    J += [(f"copies_k3_fresh_s{s}", "stages", f"variant=copies seed={s} source={{R}}/source_s{s} cuts=0.25,0.65 fresh=1 {COPY}", [f"source_s{s}"], 2000 + SCORE)]
for s in (0, 1):
    stem = f"stem_from={{R}}/plain_d4_avg_s{s} stem_model=model_avg_g8"
    # the grown model's own read is its averaged read (the averaged values are installed), so its last weights are scored
    J += [(f"tails_k1_s{s}", "stages", f"variant=tails seed={s} {stem} total=6000 cuts= score_last=0", [f"plain_d4_avg_s{s}"], 4300 + SCORE),
          (f"tails_k3_s{s}", "stages", f"variant=tails seed={s} {stem} total=6000 cuts=0.25,0.65 score_last=0", [f"plain_d4_avg_s{s}"], 4300 + SCORE),
          (f"grow_1x3_s{s}", "stages", f"variant=grow seed={s} {stem} blocks=3 steps_each=2000", [f"plain_d4_avg_s{s}"], 3600 + SCORE)]
# the class read through the middle stage; class experts and their two controls on averaged weights
J += [(f"class_to_mid_s{s}", "content", f"variant=class_to_mid seed={s} avg=g8", [], 6000 + 2 * SCORE) for s in (0, 1)]
for s in (0, 1):
    on_plain = f"seed={s} source={SRC}/plain_s{s} source_model=model_avg999 avg=g8 score_last=0"
    on_blind = f"seed={s} source={SRC}/blind_s{s} source_model=model_avg999 avg=g8 score_last=0"
    J += [(f"experts_start_avg_s{s}", "content", f"variant=experts_start {on_plain}", [], 6500 + SCORE),
          (f"rand_start_s{s}", "content", f"variant=rand_start {on_plain}", [], 6500 + SCORE),
          (f"shared_start_s{s}", "content", f"variant=shared_start {on_plain}", [], 6500 + SCORE),
          (f"blind_experts_all_avg_s{s}", "content", f"variant=blind_experts_all {on_blind}", [], 6500 + SCORE),
          (f"blind_rand_all_s{s}", "content", f"variant=blind_rand_all {on_blind}", [], 6500 + SCORE),
          (f"blind_shared_all_s{s}", "content", f"variant=blind_shared_all {on_blind}", [], 6500 + SCORE)]
# solves on averaged weights: the head, a head per stage, class heads, class heads per stage
for s in (0, 1):
    on_plain = f"seed={s} source={SRC}/plain_s{s} source_model=model_avg999 batches=200"
    on_blind = f"seed={s} source={SRC}/blind_s{s} source_model=model_avg999 batches=200"
    J += [(f"head_solved_avg_s{s}", "closed_forms", f"variant=head_solved {on_plain}", [], 500 + SCORE),
          (f"heads_staged_solved_avg_s{s}", "closed_forms", f"variant=heads_staged_solved {on_plain}", [], 900 + SCORE),
          (f"class_heads_plain_s{s}", "closed_forms", f"variant=class_heads {on_plain}", [], 500 + SCORE),
          (f"class_heads_blind_s{s}", "closed_forms", f"variant=class_heads {on_blind}", [], 500 + SCORE),
          (f"class_stage_heads_plain_s{s}", "closed_forms", f"variant=class_stage_heads {on_plain}", [], 900 + SCORE),
          (f"class_stage_heads_blind_s{s}", "closed_forms", f"variant=class_stage_heads {on_blind}", [], 900 + SCORE)]
# (phase, its jobs, whether a CPU load runs beside it, the det= of its jobs unless a job names its own)
PHASES = [("alone", ALONE, False, ""), ("loaded", LOADED, True, ""), ("fixed_loaded", FIXED_LOADED, True, "on"), ("main", J, False, "on")]
ALL = [j for _, jobs_, _, _ in PHASES for j in jobs_]
assert len({t for t, *_ in ALL}) == len(ALL), "a job tag is listed twice"
STEPS = {t: c + 900 for t, _, _, _, c in ALL}      # a job's time in reference-model steps (its training, its reads, its scoring)
RATE = 42.5                                        # the reference model's steps per second measured on this card type
total = sum(STEPS.values())
print(f"the job list: {len(ALL)} jobs, {total:,.0f} reference-model steps in all: about {total / RATE / 60:.0f} min")

The repeat check runs first, each of its three phases alone on the machine: the reference model by itself with the default settings (its loss should equal the first run's to the last digit), then the same job while every CPU of the machine is kept busy, then that again with deterministic algorithms switched on (`det=on`). By default the backward pass of the attention kernel is not deterministic on a card, so two runs of one job can differ in their last digits; with the switch on a job repeats exactly. After the three phases one runner works the job list in order, every job with `det=on`; `needs=` holds a job until the run it starts from is DONE. Each finished job folder is shipped to `REPO` under its experiment's own folder with the token from the notebook's secret `HF_TOKEN` (placed in this process's environment only; the jobs themselves run without it). When the list has ended and its report is shipped, the machine is given back (`RELEASE`), since an idle machine keeps spending.

In [ ]:
# C3 - the phases, the runner, shipping after each job, a progress line a minute, the summary, the machine given back
from geolip.modular_diffusion import jobs, hub
PREFIX = {e: f"modular_diffusion/{e}" for e in ("content", "closed_forms", "stages", "time_split")}
RELEASE = True                              # give the machine back when the list has ended and the report is shipped
RUNS = f"{OUT}/runs"
EXTRA = ""                                   # more arguments for every job (empty in a real run)
kind = {}

def line(tag, exp, args, needs, det):
    kind[tag] = exp
    a = " " + args.replace("{R}", RUNS)
    a += f" det={det}" if det and " det=" not in a else ""
    a += "" if " device=" in a else " device=cuda threads=4"
    a += "" if " score=" in a else " score=1"
    a += " reader=1" if exp != "time_split" and " score=0" not in a else ""      # the class read needs a scored job
    return f"{tag} | 60 | geolip.modular_diffusion.run | {exp}{a} data={NPZ} ref={REF}{EXTRA}" + (f" | needs={','.join(needs)}" if needs else "")

tags = [t for t, *_ in ALL]
rosters = {}
for name, jobs_, _, det in PHASES:
    rosters[name] = f"{OUT}/roster_{name}.txt"
    with open(rosters[name], "w", encoding="utf-8", newline="\n") as fh:
        fh.write("\n".join(line(t, e, a, n, det) for t, e, a, n, _ in jobs_) + "\nEND\n")
mark = lambda t, m: os.path.exists(f"{RUNS}/{t}/{m}")

if REPO:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    try:    # on a new machine: fetch this list's jobs that are already shipped and mark them done
        got_dir = snapshot_download(repo_id=REPO, allow_patterns=[f"{PREFIX[kind[t]]}/{t}/*" for t in tags], local_dir=f"{OUT}/hub_back")
        for t in tags:
            src = f"{got_dir}/{PREFIX[kind[t]]}/{t}"
            if os.path.exists(f"{src}/result.json") and not mark(t, "DONE"):
                shutil.copytree(src, f"{RUNS}/{t}", dirs_exist_ok=True)
                open(f"{RUNS}/{t}/DONE", "w").write("fetched back from the Hub\n")
    except Exception as e:
        print("nothing fetched back from the Hub:", repr(e)[:200])
for t in tags:      # a claim left by an interrupted session would hold its job forever
    if mark(t, "CLAIM") and not (mark(t, "DONE") or mark(t, "FAILED")):
        os.remove(f"{RUNS}/{t}/CLAIM")

def after_job(tag, folder, ok):
    if ok and REPO:
        hub.ship(folder, REPO, f"{PREFIX[kind[tag]]}/{tag}", f"{kind[tag].replace('_', ' ')}: {tag}")

def start_load(name):
    # every CPU of the machine kept busy by a separate process until it is stopped; returns once it is running
    out = f"{RUNS}/load_{name}"
    shutil.rmtree(out, ignore_errors=True)
    n = os.cpu_count()
    env = dict(os.environ, OMP_NUM_THREADS=str(n), MKL_NUM_THREADS=str(n), HF_TOKEN="", CUDA_VISIBLE_DEVICES="")
    p = subprocess.Popen([sys.executable, "-m", "geolip.modular_diffusion.run", "load", f"threads={n}", "seconds=1800", "size=2048", f"out={out}"],
                         env=env, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    t = time.time()
    while not (os.path.exists(f"{out}/log.txt") and os.path.getsize(f"{out}/log.txt") > 0):
        assert p.poll() is None and time.time() - t < 180, "the CPU load did not start"
        time.sleep(1)
    time.sleep(5)
    return p, n

t0 = time.time()
left0 = sum(STEPS[t] for t in tags if not (mark(t, "DONE") or mark(t, "FAILED")))
print(f"start: {sum(mark(t, 'DONE') for t in tags)}/{len(tags)} jobs already done; {left0:,.0f} steps to run")
for name, jobs_, loaded, _ in PHASES:
    if all(mark(t, "DONE") or mark(t, "FAILED") for t, *_ in jobs_):
        continue
    load = None
    if loaded:
        load, n = start_load(name)
        INSTRUMENTS["loads"][name] = {"threads": n, "started": time.strftime("%Y-%m-%d %H:%M:%S")}
    runner = threading.Thread(target=jobs.run_roster, daemon=True,
                              kwargs=dict(roster=rosters[name], out=RUNS, stop=f"{OUT}/STOP", log=f"{OUT}/roster_{name}.log", card_room_mib=None,
                                          ram_free_gb=4.0, max_hours=20.0, claim=True, after_job=after_job,
                                          env={"OMP_NUM_THREADS": "4", "MKL_NUM_THREADS": "4"}))
    runner.start()
    while runner.is_alive():
        runner.join(60)
        done, failed = sum(mark(t, "DONE") for t in tags), sum(mark(t, "FAILED") for t in tags)
        left = sum(STEPS[t] for t in tags if not (mark(t, "DONE") or mark(t, "FAILED")))
        spent = (time.time() - t0) / 60
        eta = spent * left / (left0 - left) if left0 > left else float("nan")
        print(f"jobs done {done}/{len(tags)} (failed {failed}); {spent:.0f} min spent; about {eta:.0f} min left", flush=True)
    if load is not None:
        INSTRUMENTS["loads"][name].update({"running_when_the_job_ended": load.poll() is None, "stopped": time.strftime("%Y-%m-%d %H:%M:%S")})
        load.terminate()
        try:
            load.wait(60)
        except Exception:
            load.kill()

res = lambda t: json.load(open(f"{RUNS}/{t}/result.json")) if os.path.exists(f"{RUNS}/{t}/result.json") else None
rep = {t: (res(t) or {}).get("loss_over_levels") for t in ("repeat_alone_s0", "repeat_load_s0", "ref_load_s0", "ref_s0")}
INSTRUMENTS["repeat"] = dict(rep, first_run=FIRST, nondeterministic=(res("repeat_warn_s0") or {}).get("nondeterministic"))
print("the repeat check:", json.dumps(INSTRUMENTS["repeat"], indent=1))
summary = stages.summarize(RUNS) + "\n" + content.summarize(RUNS) + "\n" + closed_forms.summarize(RUNS)
print(summary)
REPORT = f"{OUT}/report"
os.makedirs(REPORT, exist_ok=True)
open(f"{REPORT}/summary.md", "w", encoding="utf-8").write(summary)
json.dump(INSTRUMENTS, open(f"{REPORT}/instruments.json", "w", encoding="utf-8"), indent=1)   # the smoke's checks, the loads, the repeat check
for name in rosters:
    for f in (f"roster_{name}.txt", f"roster_{name}.log"):
        if os.path.exists(f"{OUT}/{f}"):
            shutil.copy(f"{OUT}/{f}", f"{REPORT}/{f.replace('.log', '_log.txt')}")
for t in tags:      # the logs of the jobs that failed (a job's own log files are not shipped with its folder)
    if mark(t, "FAILED"):
        for f in ("run.log", "log.txt"):
            if os.path.exists(f"{RUNS}/{t}/{f}"):
                shutil.copy(f"{RUNS}/{t}/{f}", f"{REPORT}/failed_{t}_{f.replace('.', '_')}.txt")
shipped = hub.ship(REPORT, REPO, "modular_diffusion/stages/_report", "stages, growth, content on averaged weights: summary and roster logs") if REPO else None
if RELEASE and REPO and shipped:
    from google.colab import runtime
    runtime.unassign()